In [1]:
1+1

2

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("dataset.csv")

print(df.shape)
df.head()


(975, 30)


,datetime,underlying_price,NIFTY27JAN2625200CE,NIFTY27JAN2625300CE,NIFTY27JAN2625400CE,NIFTY27JAN2625500CE,NIFTY27JAN2625600CE,NIFTY27JAN2625700CE,NIFTY27JAN2625800CE,NIFTY27JAN2625900CE,...,NIFTY27JAN2624200PE,NIFTY27JAN2624300PE,NIFTY27JAN2624400PE,NIFTY27JAN2624500PE,NIFTY27JAN2624600PE,NIFTY27JAN2624700PE,NIFTY27JAN2624800PE,NIFTY27JAN2624900PE,NIFTY27JAN2625000PE,NIFTY27JAN2625100PE
0,07-01-2026 09:15,26111.65,0.12662,0.12330,0.11741,NaN,0.11005,0.10576,NaN,0.09724,...,0.15760,0.15240,0.14697,0.14105,0.13613,0.13085,0.12640,0.12142,0.11631,0.11150
1,07-01-2026 09:20,26141.40,0.08632,NaN,NaN,0.11779,0.11197,0.11028,NaN,NaN,...,NaN,0.15420,0.14753,0.14274,0.13849,0.13282,NaN,0.12363,NaN,0.11353
2,07-01-2026 09:25,26139.35,0.09147,NaN,0.09514,0.09933,0.09599,0.09204,0.09216,0.08954,...,0.15927,NaN,0.14919,0.14245,0.13806,0.13242,0.12877,0.12349,0.11817,NaN
3,07-01-2026 09:30,26128.95,0.10860,0.10842,0.11150,0.12248,0.10715,0.11098,0.10345,NaN,...,0.15755,NaN,0.14691,0.14209,0.13721,0.13184,0.12722,0.12252,0.11729,0.11200
4,07-01-2026 09:35,26131.90,0.10462,0.10538,0.12459,0.12051,0.11225,0.11294,0.10544,NaN,...,0.15924,0.15334,0.14784,0.14230,NaN,0.13219,0.12733,0.12295,0.11707,NaN


In [3]:
iv_cols = [c for c in df.columns if c != "datetime"]

print(len(iv_cols))
print(iv_cols[:5])

29
['underlying_price', 'NIFTY27JAN2625200CE', 'NIFTY27JAN2625300CE', 'NIFTY27JAN2625400CE', 'NIFTY27JAN2625500CE']


In [4]:
iv_cols = [
    c for c in df.columns
    if c not in ["datetime", "underlying_price"]
]

print(len(iv_cols))
print(iv_cols[:5])

28
['NIFTY27JAN2625200CE', 'NIFTY27JAN2625300CE', 'NIFTY27JAN2625400CE', 'NIFTY27JAN2625500CE', 'NIFTY27JAN2625600CE']


In [5]:
matrix = df[iv_cols].copy()

print(matrix.shape)

(975, 28)


In [6]:
print(matrix.isna().sum().sum())

5460


In [7]:
filled = matrix.copy()

for col in filled.columns:
    filled[col] = filled[col].fillna(
        filled[col].mean()
    )

print(filled.isna().sum().sum())

0


In [8]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(
    n_components=10,
    random_state=42
)

Z = svd.fit_transform(filled)

reconstructed = svd.inverse_transform(Z)

reconstructed = pd.DataFrame(
    reconstructed,
    columns=iv_cols
)

print(reconstructed.shape)

(975, 28)


In [9]:
filled_df = df.copy()

for col in iv_cols:

    mask = filled_df[col].isna()

    filled_df.loc[
        mask,
        col
    ] = reconstructed.loc[
        mask,
        col
    ]

In [10]:
print(
    filled_df[iv_cols]
    .isna()
    .sum()
    .sum()
)

0


In [11]:
filled_df.to_csv(
    "filled_dataset_svd.csv",
    index=False
)

print("filled_dataset_svd.csv saved")

filled_dataset_svd.csv saved


In [12]:
import pandas as pd

ORIGINAL_DATASET_PATH = "dataset.csv"

SEPARATOR = "||"

def generate_solution(
    filled_path,
    output_path="submission_svd.csv"
):

    original = pd.read_csv(
        ORIGINAL_DATASET_PATH
    )

    filled = pd.read_csv(
        filled_path
    )

    feature_cols = [
        c for c in original.columns
        if c != "datetime"
        and c != "underlying_price"
    ]

    rows = []

    for col in feature_cols:

        was_missing = original[col].isna()

        for idx in original.index[was_missing]:

            dt = original.loc[idx, "datetime"]

            uid = f"{dt}{SEPARATOR}{col}"

            val = filled.loc[idx, col]

            rows.append({
                "id": uid,
                "value": val
            })

    solution = pd.DataFrame(rows)

    solution = (
        solution
        .sort_values("id")
        .reset_index(drop=True)
    )

    solution.to_csv(
        output_path,
        index=False
    )

    print(
        output_path,
        solution.shape
    )

In [13]:
generate_solution(
    "filled_dataset_svd.csv",
    "submission_svd.csv"
)

submission_svd.csv (5460, 2)


In [14]:
sub = pd.read_csv(
    "submission_svd.csv"
)

print(sub.shape)
print(sub.head())

(5460, 2)
                                      id     value
0  07-01-2026 09:15||NIFTY27JAN2624100PE  0.213549
1  07-01-2026 09:15||NIFTY27JAN2625500CE  0.108380
2  07-01-2026 09:15||NIFTY27JAN2625800CE  0.116509
3  07-01-2026 09:20||NIFTY27JAN2624000PE  0.260238
4  07-01-2026 09:20||NIFTY27JAN2624200PE  0.214626


In [15]:
print(reconstructed.describe())

print(
    reconstructed.min().min(),
    reconstructed.max().max()
)

       NIFTY27JAN2625200CE  NIFTY27JAN2625300CE  NIFTY27JAN2625400CE  \
count           975.000000           975.000000           975.000000   
mean              0.138384             0.136829             0.139053   
std               0.057606             0.066520             0.072864   
min               0.072861             0.090412             0.096209   
25%               0.117522             0.114051             0.115109   
50%               0.124736             0.119422             0.121065   
75%               0.132258             0.125032             0.127208   
max               0.575207             0.632997             0.732123   

       NIFTY27JAN2625500CE  NIFTY27JAN2625600CE  NIFTY27JAN2625700CE  \
count           975.000000           975.000000           975.000000   
mean              0.140259             0.146152             0.153555   
std               0.096212             0.120805             0.142609   
min               0.093175             0.084252             0.0

In [16]:
reconstructed_clipped = reconstructed.clip(
    lower=0.05,
    upper=0.50
)

print(
    reconstructed_clipped.min().min(),
    reconstructed_clipped.max().max()
)

0.05 0.5


In [17]:
filled_df = df.copy()

for col in iv_cols:

    mask = filled_df[col].isna()

    filled_df.loc[
        mask,
        col
    ] = reconstructed_clipped.loc[
        mask,
        col
    ]

In [19]:
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import TruncatedSVD

# Original matrix
mat = matrix.copy()

# Locations with known values
known_positions = np.argwhere(~mat.isna().values)

# Hide 20% randomly
np.random.seed(42)

test_idx = np.random.choice(
    len(known_positions),
    size=int(0.2 * len(known_positions)),
    replace=False
)

masked = mat.copy()

true_values = []
test_positions = []

for idx in test_idx:

    r, c = known_positions[idx]

    true_values.append(mat.iloc[r, c])

    test_positions.append((r, c))

    masked.iloc[r, c] = np.nan

# Mean fill
filled_masked = masked.copy()

for col in filled_masked.columns:
    filled_masked[col] = filled_masked[col].fillna(
        filled_masked[col].mean()
    )

# SVD
svd = TruncatedSVD(
    n_components=10,
    random_state=42
)

Z = svd.fit_transform(filled_masked)

recon = svd.inverse_transform(Z)

recon = pd.DataFrame(
    recon,
    columns=iv_cols
)

# Predictions
preds = []

for r, c in test_positions:
    preds.append(recon.iloc[r, c])

mse = mean_squared_error(
    true_values,
    preds
)

print("SVD Validation MSE =", mse)
print("SVD validation phase as the result of the SVD imputation on the masked dataset.")

SVD Validation MSE = 0.03993368520024864
SVD validation phase as the result of the SVD imputation on the masked dataset.
